## RAG - Routing
In the [Query Translation](02_Query_Translation.ipynb) notebook we covered several techniques for translating user's query - techniques such as Multi-Query, RAG Fusion, Query-Decomposition, Step Back and HYDE were discussed. The goal such techniques is to take input user question and translate it in such a way as to improve retrieval. In this notebook, we'll discuss the next step in the RAG pipeline - _Routing_.

<center>
<img src="images/rag_routing.png" width="800" height="480"/>
</center>

### What is Routing?
**Routing** is the next step, which is potentially _directing_ the translated user query to the right source. We could have several sources of data that the user would like to query from, sources such as vector-stores,  GraphDBs or an RDBMS. We simply route (or direct) the query to the right source based upon content of the question. There are a few different ways to do that.

One of the techniques is called **Logical Routing**. In this case we basically give our LLM knowledge of the various datasources that we have at our disposal and we let the LLM _reason_ about which one to apply the question to.

<center>
<img src="images/logical_routing.png"/>
</center>

Alternatively, we could use **Semantic Routing**, which is where we take take the user's question/query, we embed it, we also embed prompts and compare the similarity betweeen our question and embedded prompts and we choose a prompt based on the similarity.

<center>
<img src="images/semantic_routing.png"/>
</center>

So the general idea is to route question to different prompts (or arbitrarily taking the question and sending it to the rioht source that can answer it).

In this notebook, we will cover both _Logical_ and _Semantic_ routing techniques. We'll be using the LangChain framework with Google Gemini 2.5 Flash LLM, but you can always replace it with an LLM of your choice below. 

In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

os.environ["USER_AGENT"] = "LangChain_AdvRagTech/1.0"

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# as with the Naive RAG notebook, we'll use OpenAI LLM and OpenAI Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# and ChromaDB as our vector store
from langchain_community.vectorstores import Chroma

In [2]:
# load API keys from .env files
load_dotenv(override=True)
console = Console()

In [3]:
# create our LLM - we will be using OpenAI GPT 3.5 Turbo (fairly cheap)
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)

### Logical Routing

A business scenario that could explain the need for Logical routing would be something like the following: 

Let's say _WeCoverAnything_ (WCA) a [fictitious] Insurance company tha sells Life Insurance, Health Insurance and Car Insurance policies. Bob is a client who has bought Health _and_ Car insurance policies from WCA. WCA has deployed a customer facing chatbot (powered by an LLM of course!) that can answer common questions from end-customers such as Bob (or redirect their query to human Helpdesk agents in case it cannot answer the question). Bob could ask a question like "Is my policy covering both own damage and third-party liability? Can you explain what each one means?" or something like "I need to file a claim for a recent hospitalization. What documents do I need, and how do I submit them?". Clearly the first relates to Car Insurance and the second to Health Insurance. The chabot must be intelligent enough to direct the former question to "Car Insurance Policy" datasources and the latter to "Health Insurance Policy" data sources.

Let's walk through a Logical Routing use-case. Suppose that we have 3 knowledge sources focused related to Python, JavaScript and Go programming respectively. So, we'd like to re-direct all Python queries to the Python datastore, JavaScript to the JS data store and so on. 

<center>
<img src="images/logical_routing.png"/>
</center>

We bind LLM output to a structure (a Pydantic datamodel), so that it returns one of a fixed set of outputs, which we can then use to redirect to specific datasource. We use the `llm_with_structured_output(...)` call to bind LLM's output structure to the defined Pydantic class.

In [4]:
from typing import Literal
from pydantic import BaseModel, Field


class UserQueryRouter(BaseModel):
    """Route a user query regarding insurance policy to car or health policy sources"""

    datasource: Literal["car_policy_docs", "health_policy_docs", "unknown"] = Field(
        ...,
        description="""Given a user question regarding his/her car or health insurance policy 
        choose which datasource would be most relevant for answering their policy related 
        question. If the question is not related to either car or health insurance, 
        return 'unknown'""",
    )


structured_llm_policy = llm.with_structured_output(UserQueryRouter)


# Data model: here we define various "routes" depending on programming language
# So Python related queries should go to "python_docs", JavaScript to "js_docs" etc.
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    # these could be paths of vector stores, or identifiers for APIs etc.
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="""Given a user question choose which datasource would be most relevant 
        for answering their question""",
    )


structured_llm_code = llm.with_structured_output(RouteQuery)

In [6]:
# Prompt
system_prompt_policy = """You are an expert at routing a user question to the appropriate data source.

Based on the type of insurance policy (Car or Health) the question is referring to, route it to the 
relevant data source. If it is referring to neither, route it to 'unknown'."""

from langchain_core.prompts import ChatPromptTemplate

policy_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_policy),
        ("human", "{question}"),
    ]
)

# Define router
policy_router = policy_prompt | structured_llm_policy

In [8]:
# now let's try it out with a question specific to Car insurance
question_car_insurance = """
    Is my policy covering both own damage and third-party liability? 
    Can you explain what each one means?
"""

# NOTE: since the router is bound to UserQueryRouter class, an instance
# of this class is returned, rather than plan or markedup text.
result = policy_router.invoke({"question": question_car_insurance})
print(result.datasource)

car_policy_docs


In [9]:
# now let's try it out with a question specific to Health Insurance
question_health_insurance = """
    I need to file a claim for a recent hospitalization. What documents do I need, 
    and how do I submit them?
"""

result = policy_router.invoke({"question": question_health_insurance})
print(result.datasource)

health_policy_docs


In [ ]:
# what if I ask an unrelated (to car or healh insurance) question?
question_random = """
    My laptop is freezing randomly. I just upgraded by RAM. Could it be a RAM issue?
"""

# the above is neither a car insurnace nor a health insurance query
# so expect an "unknown" response. In real-time, this could be a signal
# to re-route the call to a Human attendant.
result = policy_router.invoke({"question": question_random})
print(result.datasource)

unknown


In [15]:
# simulates a "real" function that can handle both health & car insurance
# queries, but escalate others to human attendants
def choose_policy_route(result):
    if "car_policy_docs" in result.datasource.lower():
        # code below would be the actual car insurance handling logic
        # Say a call to car-insurance RAG pipeline's vector store
        return "chain for handling Car Insurance related query"
    elif "health_policy_docs" in result.datasource.lower():
        # code below would be the actual health insurance handling logic
        # Say a call to car-insurance RAG pipeline's vector store
        return "chain for handling Health Insurance related query"
    else:
        # code that will escalate this query to human agent
        return (
            "Aplogies, I don't understand your question - routing to a human advisor."
        )


from langchain_core.runnables import RunnableLambda

full_policy_chain = policy_router | RunnableLambda(choose_policy_route)

In [12]:
full_policy_chain.invoke({"question": question_car_insurance})

'chain for handling Car Insurance related query'

In [16]:
full_policy_chain.invoke({"question": question_health_insurance})

'chain for handling Health Insurance related query'

In [17]:
# try out with any question
# full_policy_chain.invoke({"question": "Do you cove headlight repairs?"})
full_policy_chain.invoke({"question": "Do you cover cyclone damage of my home?"})

"Aplogies, I don't understand your question - routing to a human advisor."

In [18]:
full_policy_chain.invoke({"question": question_random})

"Aplogies, I don't understand your question - routing to a human advisor."

In [19]:
# Prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router
router = prompt | structured_llm_code

In [20]:
# now let's try it out with various languages
# Python first
question_python = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question_python})
print(result.datasource)

python_docs


In [21]:
# How about this?
question_go = """Why doesn't the following code work:

import (
	"fmt"
)

func main() {
	m := make(map[string]int)
	vals := []int{1, 2, 3}

	for _, v := range vals {
		go func() {
			m["sum"] += v
		}()
	}

	fmt.Println("sum:", m["sum"])
}
"""

result = router.invoke({"question": question_go})
print(result.datasource)

golang_docs


In [22]:
# And this?
question_js = """Why doesn't the following code work:

let count = 0;

for (var i = 0; i < 5; i++) {
  setInterval(function () {
    count++;
    console.log("i:", i, "count:", count);
    if (count === 5) {
      clearInterval(this);
    }
  }, 1000);
}
"""

result = router.invoke({"question": question_js})
print(result.datasource)

js_docs


So as you can see, the LLM is able to _detect_ the programming language and _direct_ us to the correct data source to redirect our query to!

Now let us create a common function to route.

In [23]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        ### Logic here
        return "chain for js_docs"
    else:
        ### Logic here
        return "golang_docs"


from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)

In [24]:
full_chain.invoke({"question": question_js})

'chain for js_docs'

In [25]:
full_chain.invoke({"question": question_python})

'chain for python_docs'

In [26]:
full_chain.invoke({"question": question_go})

'golang_docs'

### Semantic Routing

Semantic routing is a little bit straightforward as compared to Logical Routing. In this case we have a series of prompts (or more correctly, prompt templates) we want to choose depending on the input query - so say. we have a Physics related prompt-template and a Math related prompt-template and the user asks a question related to either Physics or Maths. 

First we embed both the templates as well as the user's question. Then we use a `cosine similarity` function to choose which subject the prompt is related to, and then fire the "closest match" prompt.

<center>
<img src="images/semantic_routing.png"/>
</center>

In [35]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser

In [36]:
# Two prompt templates for different subjects
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

In [37]:
embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

In [38]:
# Route question to correct prompt
def prompt_router(input):
    # Embed question
    query_embedding = embeddings.embed_query(input["query"])
    # Compute similarity between embedded query and embedded prompt templates
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    print(f"Similarities -> {similarity}")
    most_similar = prompt_templates[similarity.argmax()]
    # Chosen prompt
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

In [39]:
prompt_router({"query": "What is Newton's second law of motion?"})  # a Physics question

Similarities -> [0.75716666 0.72733265]
Using PHYSICS


PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template="You are a very smart physics professor. You are great at answering questions about physics in a concise and easy to understand manner. When you don't know the answer to a question you admit that you don't know.\n\nHere is a question:\n{query}")

In [40]:
# now let's build our chain
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

In [41]:
chain.invoke("What is Newton's second law of motion?")  # Physics question

Similarities -> [0.75716666 0.72733265]
Using PHYSICS


"Newton's second law says that the net external force on a body (or system) equals the time rate of change of its momentum:\n- F_net = dp/dt\n- where p = m v is the momentum.\n\nIf the mass is constant, this reduces to the familiar form:\n- F_net = m a\n\nFor systems where mass can change (like a rocket), the general form is:\n- F_net = d(m v)/dt = m dv/dt + v dm/dt\n\nUnits: 1 newton (N) = 1 kg·m/s^2.\n\nExample: A 2 kg object accelerated at 3 m/s^2 requires a net force of 6 N (F = m a)."

In [42]:
chain.invoke("What is Pythagoras' theorem?")  # Math question

Similarities -> [0.76623209 0.78525428]
Using MATH


'- Statement: In a Euclidean right triangle, if a and b are the lengths of the two legs and c is the length of the hypotenuse (the side opposite the right angle), then a^2 + b^2 = c^2.\n\n- Quick example: For a 3-4-5 triangle, 3^2 + 4^2 = 9 + 16 = 25 = 5^2, so the hypotenuse is 5.\n\n- Extra note: The converse is true—if a triangle with side lengths a, b, c satisfies a^2 + b^2 = c^2, then the triangle is right-angled opposite side c.'

In [43]:
# Math or Physics?
chain.invoke(
    "If a car travels at a constant speed of 60 miles per hour, how far will it travel in 3 hours?"
)

Similarities -> [0.72781209 0.72725169]
Using PHYSICS


'Distance = speed × time. At 60 mph for 3 hours, the car travels 60 × 3 = 180 miles. (Approximately 290 km if you prefer metric: 180 miles × 1.609 ≈ 289.7 km.)'

In [44]:
chain.invoke(
    "What is the maximum volume of a sphere that can be contained within a cube of side length $L$, and how does the pressure exerted by an ideal gas inside that sphere relate to its temperature?"
)

Similarities -> [0.74111346 0.72604615]
Using PHYSICS


'- The largest sphere that fits inside a cube of side length L is the inscribed sphere, with diameter L. So its radius is r = L/2.\n\n- Its maximum volume is\n  V_max = (4/3)πr^3 = (4/3)π(L/2)^3 = π L^3 / 6.\n\n- If an ideal gas with n moles (or N particles) is confined inside this sphere, the pressure and temperature are related by the ideal gas law PV = nRT (or P = NkT/V).\n\n- Since the gas is confined to the fixed sphere volume V = π L^3 / 6, the pressure scales linearly with temperature:\n  P = nRT / V = (6 nR / (π L^3)) T\n  (or P = NkT / V with N particles).\n\nSo the maximum sphere volume is πL^3/6, and the pressure inside the sphere is proportional to temperature for fixed amount of gas.'

In [45]:
from pydantic import BaseModel, Field


class InputQuestionsList(BaseModel):
    """Input schema for list of questions."""

    num_questions: int = Field(..., description="Number of user questions.")
    questions: list[str] = Field(..., description="List of user questions.")


structured_llm = llm.with_structured_output(InputQuestionsList)

In [46]:
prompt_template = """
From the following user input, extract the number questions asked, and extract the questions separately. User could ask multiple questions in a single input.

User Input: {user_input}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert at extracting questions from user input."),
        ("human", prompt_template),
    ]
)

question_chain = prompt | structured_llm

In [47]:
response = question_chain.invoke(
    {
        "user_input": "What is Newton's second law of motion? Also, what is Pythagoras' theorem?"
    }
)
print(f"I extracted {response.num_questions} questions.")
print("Here are the questions:")
for i in range(response.num_questions):
    print(f"{i+1} -> {response.questions[i]}")

I extracted 2 questions.
Here are the questions:
1 -> What is Newton's second law of motion?
2 -> What is Pythagoras' theorem?
